In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!gunzip /content/drive/MyDrive/Capstone/meta_Clothing_Shoes_and_Jewelry.jsonl.gz


In [ ]:
!ls /content/drive/MyDrive/Capstone/meta_Clothing_Shoes_and_Jewelry.jsonl

In [ ]:
!pip install duckdb


In [ ]:
import duckdb


In [ ]:
con = duckdb.connect(
    '/content/drive/MyDrive/Capstone/comemo.db'
)


In [ ]:
# Run metadata slim table SQL
with open('/content/drive/MyDrive/Capstone/Comemo-Dataset/sql/02_slim_tables.sql', 'r') as f:
    sql_query = f.read()

# Modify the SQL query to include ignore_errors=true and TRY_CAST for average_rating
# Use a more robust replacement for 'ignore_errors' by targeting a specific part within read_json
sql_query = sql_query.replace(
    "format='newline_delimited'",
    "format='newline_delimited',\n    ignore_errors=true"
)

# Replace average_rating with TRY_CAST to handle non-numeric values gracefully
sql_query = sql_query.replace(
    "average_rating",
    "TRY_CAST(average_rating AS DOUBLE) AS average_rating"
)

con.execute(sql_query)

In [ ]:
con.execute("SHOW TABLES").fetchdf()



In [ ]:
con.execute("SELECT COUNT(*) FROM metadata_raw").fetchone()


In [ ]:
con.execute("""
SELECT price_raw
FROM metadata_raw
WHERE price_raw IS NOT NULL
LIMIT 10
""").fetchdf()


In [ ]:
# Run reviews load SQL
with open('/content/drive/MyDrive/Capstone/Comemo-Dataset/sql/01_load_json.sql', 'r') as f:
    sql_query = f.read()

# Modify the SQL query for metadata_raw table to handle errors and casting
# First, add ignore_errors=true to the read_json for metadata.jsonl
# This requires a more targeted replacement
metadata_json_read_pattern = """FROM read_json(
    '/content/drive/MyDrive/Capstone/comemo_data/metadata.jsonl',
    format='newline_delimited'
)"""
metadata_json_read_replacement = """FROM read_json(
    '/content/drive/MyDrive/Capstone/comemo_data/metadata.jsonl',
    format='newline_delimited',
    ignore_errors=true
)"""
sql_query = sql_query.replace(metadata_json_read_pattern, metadata_json_read_replacement)

# Second, replace average_rating with TRY_CAST to handle non-numeric values gracefully
# This replacement is assumed to target the average_rating column in the metadata_raw SELECT statement.
sql_query = sql_query.replace(
    "average_rating",
    "TRY_CAST(average_rating AS DOUBLE) AS average_rating"
)

print("--- SQL Query to be executed ---")
print(sql_query)
print("--------------------------------")

con.execute(sql_query)

In [ ]:
con.execute("SHOW TABLES").fetchdf()

### Persisting the DuckDB database

Although DuckDB automatically writes changes to the database file when connected, explicitly closing the connection can help ensure all data is flushed to disk, especially in environments like Colab. After you've created all your tables, you can close the connection.

In [ ]:
con.close()
print("DuckDB connection closed. Tables should be persisted to /content/drive/MyDrive/Capstone/comemo.db")

### Verifying persistence after runtime restart

After running the above cell and then restarting your Colab runtime (or disconnecting and reconnecting the backend), you can verify that the tables are still present by:
1. Re-running the cell to mount your Google Drive (`aN26xuEY1pKy`).
2. Re-running the cells to import `duckdb` and connect to your database (`biFDJrIn7YAf` and `YzkW9L7n7Y0D`).
3. Running `con.execute("SHOW TABLES").fetchdf()` again. You should see `metadata_raw` and `reviews_raw` listed.

To reconnect to the DuckDB database, simply re-run the cells that import `duckdb` and establish the connection to your database file.

In [ ]:
import duckdb

con = duckdb.connect(
    '/content/drive/MyDrive/Capstone/comemo.db'
)

print("DuckDB connection re-established.")

In [ ]:
con.execute("""
COPY reviews_raw
TO '/content/drive/MyDrive/Capstone/reviews_raw.parquet'
(FORMAT PARQUET);

COPY metadata_raw
TO '/content/drive/MyDrive/Capstone/metadata_raw.parquet'
(FORMAT PARQUET);
""")

In [ ]:
con.execute("SHOW TABLES").fetchdf()